Training

In [ ]:
# =============================================================================
# Complete Experimental Code for GPT-2 Large (0.77B) with LoRA on Code Alpaca
# Using Cosine Annealing with Warm Restarts & Doubling Initial LR Each Cycle
# Adaptive column names to avoid KeyError.
# All comments in English.
# =============================================================================

# 1. Fix version conflicts: uninstall problematic packages, set env, install required packages
!pip uninstall torchvision torchaudio -y
import os
os.environ["DATASETS_USE_TORCHVISION"] = "0"   # Prevents transformers from trying to use torchvision

# Install necessary packages; ensure torchao >= 0.16.0 for peft compatibility
!pip install -q transformers datasets peft accelerate torch matplotlib "torchao>=0.16.0"

# 2. Import libraries
import math
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    default_data_collator,
)
from torch.optim.lr_scheduler import LambdaLR
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 3. Global configuration
# ------------------------------------------------------------
MODEL_NAME = "gpt2-large"               # 0.77B parameters
DATASET_NAME = "HuggingFaceH4/CodeAlpaca_20K"
MAX_SEQ_LEN = 1024
BATCH_SIZE = 4
TOTAL_STEPS = 20000
CHECKPOINT_INTERVAL = 2000
RECORD_INTERVAL = 20
VAL_INTERVAL = 500
SEED = 42
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
BASE_LR = 2e-4                         # initial learning rate (after warmup)
WARMUP_STEPS = 100
CYCLE_LEN = 4500                       # restart every 4500 steps

# ------------------------------------------------------------
# 4. Set seed and device
# ------------------------------------------------------------
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------------------------------------
# 5. Load dataset, tokenizer, and adaptive tokenization
# ------------------------------------------------------------
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def detect_columns(examples):
    """Detect which column names are present and return a mapping."""
    if 'instruction' in examples:
        return {'instruction': 'instruction', 'input': 'input', 'output': 'output'}
    elif 'prompt' in examples:
        return {'instruction': 'prompt', 'input': 'input', 'output': 'completion'}
    elif 'question' in examples:
        return {'instruction': 'question', 'input': 'input', 'output': 'answer'}
    else:
        # If nothing matches, raise an error showing available columns
        raise KeyError(f"Unknown columns. Available: {list(examples.keys())}")

def tokenize_function(examples):
    mapping = detect_columns(examples)
    inst_col = mapping['instruction']
    inp_col = mapping.get('input')
    out_col = mapping.get('output')

    insts = examples[inst_col]
    inputs = examples.get(inp_col, [''] * len(insts)) if inp_col else [''] * len(insts)
    outputs = examples.get(out_col, [''] * len(insts)) if out_col else [''] * len(insts)

    texts = []
    for inst, inp, out in zip(insts, inputs, outputs):
        if inp and len(inp.strip()) > 0:
            text = f"{inst}\n{inp}\n{out}"
        else:
            text = f"{inst}\n{out}"
        texts.append(text)
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        return_tensors='pt'
    )

# Load dataset and split
full_dataset = load_dataset(DATASET_NAME, split='train')
full_dataset = full_dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = full_dataset['train']
valid_dataset = full_dataset['test']

print(f"Train size: {len(train_dataset)}, Valid size: {len(valid_dataset)}")
print("Columns in train dataset:", train_dataset.column_names)   # debug

# Tokenize (remove original columns)
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)
valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
valid_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
    drop_last=True
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=default_data_collator
)

steps_per_epoch = len(train_loader)
print(f"Steps per epoch: {steps_per_epoch}")

# ------------------------------------------------------------
# 6. Helper functions for physics recording
# ------------------------------------------------------------
def get_flat_params(model):
    """Return a 1D tensor of all trainable parameters (on CPU)."""
    with torch.no_grad():
        params = [p.data.view(-1) for p in model.parameters() if p.requires_grad]
        if not params:
            return torch.tensor([0.0])
        return torch.cat(params).cpu()

def evaluate_loss(model, data_loader):
    """Evaluate loss on a subset of validation data."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch['input_ids'])
            total_loss += outputs.loss.item()
            count += 1
            if count >= 20:   # limit to 20 batches for speed
                break
    model.train()
    return total_loss / count if count > 0 else float('inf')

# ------------------------------------------------------------
# 7. Main training function
# ------------------------------------------------------------
def train_with_cosine_annealing_restarts():
    """
    Train GPT-2 Large with LoRA and a custom Cosine Annealing with Warm Restarts
    where the initial LR of each cycle doubles compared to the previous cycle.
    Warmup only in the first cycle (first 100 steps).
    """
    # --- Load base model ---
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
    model = model.to(device)

    # --- Configure LoRA ---
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=['c_attn', 'c_proj'],
        bias='none',
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # --- Optimizer ---
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LR,          # will be scaled by the lambda
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01
    )

    # --- Custom scheduler: Cosine Annealing with Warm Restarts & Growing initial LR ---
    def lr_lambda(step):
        # step is zero-based
        if step < WARMUP_STEPS:
            # Linear warmup from 0 to BASE_LR
            return step / WARMUP_STEPS

        # Determine cycle index and position within cycle
        cycle = step // CYCLE_LEN
        t = (step % CYCLE_LEN) / CYCLE_LEN   # progress in [0, 1) within the cycle

        if cycle == 0:
            # First cycle: warmup + cosine
            # After warmup, we want cosine from BASE_LR down to 0.
            # The warmup took WARMUP_STEPS, so the remaining part is (CYCLE_LEN - WARMUP_STEPS)
            if t < WARMUP_STEPS / CYCLE_LEN:
                # This should not happen for step >= WARMUP_STEPS, but for safety:
                return t / (WARMUP_STEPS / CYCLE_LEN)
            else:
                # Cosine annealing from BASE_LR down to 0
                t_cos = (t - WARMUP_STEPS / CYCLE_LEN) / (1 - WARMUP_STEPS / CYCLE_LEN)
                return 0.5 * (1 + math.cos(math.pi * t_cos))
        else:
            # Subsequent cycles: no warmup, cosine from 2^cycle * BASE_LR down to 0
            base_multiplier = 2 ** cycle   # e.g., cycle=1 -> 2, cycle=2 -> 4, ...
            return base_multiplier * 0.5 * (1 + math.cos(math.pi * t))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    # --- Training state ---
    step = 0
    epoch = 0
    losses = []
    val_losses = []
    val_steps = []
    recorded_steps = []
    learning_rates = []
    re_old_raw = []
    re_old_ema = []
    re_new_raw = []
    re_new_ema = []
    v_norm_list = []
    a_norm_list = []
    mass_list = []
    gamma_list = []

    param_queue = []
    ema_re_old = None
    ema_re_new = None
    ema_alpha = 0.9

    pbar = tqdm(total=TOTAL_STEPS, desc="Training", position=0, leave=True)

    # Training loop
    while step < TOTAL_STEPS:
        for batch in train_loader:
            if step >= TOTAL_STEPS:
                break

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch, labels=batch['input_ids'])
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            losses.append(loss.item())

            # --- Record physics every RECORD_INTERVAL steps ---
            if step % RECORD_INTERVAL == 0:
                flat = get_flat_params(model)
                param_queue.append(flat)
                if len(param_queue) > 3:
                    param_queue.pop(0)

                recorded_steps.append(step)
                current_lr = scheduler.get_last_lr()[0]
                learning_rates.append(current_lr)

                if len(param_queue) >= 3:
                    v_k = param_queue[1] - param_queue[0]
                    v_k1 = param_queue[2] - param_queue[1]
                    a_k = v_k1 - v_k

                    norm_v = torch.norm(v_k).item()
                    norm_a = torch.norm(a_k).item()
                    v_norm_list.append(norm_v)
                    a_norm_list.append(norm_a)

                    re_old = norm_a / (norm_v + 1e-12)
                    re_old_raw.append(re_old)
                    if ema_re_old is None:
                        ema_re_old = re_old
                    else:
                        ema_re_old = ema_alpha * ema_re_old + (1 - ema_alpha) * re_old
                    re_old_ema.append(ema_re_old)

                    # Compute mass and gamma from Adam state
                    if isinstance(optimizer, torch.optim.AdamW):
                        total_sq = 0.0
                        count = 0
                        for group in optimizer.param_groups:
                            for p in group['params']:
                                state = optimizer.state[p]
                                if 'exp_avg_sq' in state:
                                    total_sq += torch.mean(state['exp_avg_sq']).item()
                                    count += 1
                        if count > 0:
                            avg_sq = total_sq / count
                            eps = 1e-8
                            eta_eff = current_lr / (math.sqrt(avg_sq) + eps)
                            beta1 = 0.9
                            m_current = 1.0 / (eta_eff * (1 - beta1))
                            mass_list.append(m_current)
                            if len(mass_list) >= 2:
                                gamma_current = mass_list[-2] - beta1 * m_current
                            else:
                                gamma_current = 0.0
                            gamma_list.append(gamma_current)
                        else:
                            mass_list.append(0.0)
                            gamma_list.append(0.0)
                    else:
                        mass_list.append(0.0)
                        gamma_list.append(0.0)

                    if len(mass_list) > 0 and len(gamma_list) > 0:
                        m_val = mass_list[-1]
                        g_val = gamma_list[-1]
                        re_new = (m_val * norm_a) / (g_val * norm_v + 1e-12)
                    else:
                        re_new = 0.0
                    re_new_raw.append(re_new)
                    if ema_re_new is None:
                        ema_re_new = re_new
                    else:
                        ema_re_new = ema_alpha * ema_re_new + (1 - ema_alpha) * re_new
                    re_new_ema.append(ema_re_new)

                pbar.set_description(f"Loss: {loss.item():.4f} | LR: {current_lr:.2e}")

            # --- Validation ---
            if step % VAL_INTERVAL == 0 and step > 0:
                val_loss = evaluate_loss(model, valid_loader)
                val_losses.append(val_loss)
                val_steps.append(step)

            # --- Checkpoint ---
            if step % CHECKPOINT_INTERVAL == 0 and step > 0:
                checkpoint_dir = f"checkpoint_step_{step}"
                os.makedirs(checkpoint_dir, exist_ok=True)
                model.save_pretrained(checkpoint_dir)
                torch.save({
                    'step': step,
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                }, os.path.join(checkpoint_dir, 'training_state.pt'))
                print(f"\nCheckpoint saved at step {step}")

            step += 1
            pbar.update(1)

        epoch += 1
        if step >= TOTAL_STEPS:
            break

    pbar.close()
    # Final checkpoint
    final_dir = "checkpoint_final"
    os.makedirs(final_dir, exist_ok=True)
    model.save_pretrained(final_dir)
    torch.save({
        'step': step,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }, os.path.join(final_dir, 'training_state.pt'))
    print("Final checkpoint saved.")

    return {
        'steps': recorded_steps,
        'train_loss': losses,
        'val_loss': val_losses,
        'val_steps': val_steps,
        'learning_rates': learning_rates,
        'Re_old_raw': re_old_raw,
        'Re_old_ema': re_old_ema,
        'Re_new_raw': re_new_raw,
        'Re_new_ema': re_new_ema,
        'v_norm': v_norm_list,
        'a_norm': a_norm_list,
        'mass_history': mass_list,
        'gamma_history': gamma_list,
    }

# ------------------------------------------------------------
# 8. Run experiment and save results
# ------------------------------------------------------------
print("Starting training with Cosine Annealing with Warm Restarts (doubling initial LR each cycle)...")
exp_data = train_with_cosine_annealing_restarts()

np.savez("experiment_results.npz", **exp_data)
with open("experiment_results.pkl", "wb") as f:
    pickle.dump(exp_data, f)

print("Experiment completed. Results saved.")

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.9 MB/s eta 0:00:00


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 4,055,040 || all params: 778,085,120 || trainable%: 0.5212


Training:   0%|          | 0/20000 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Checkpoint saved at step 2000

Checkpoint saved at step 4000

Checkpoint saved at step 6000

Checkpoint saved at step 8000

Checkpoint saved at step 10000

Checkpoint saved at step 12000

Checkpoint saved at step 14000

Checkpoint saved at step 16000

Checkpoint saved at step 18000
Final checkpoint saved.
Experiment completed. Results saved.
